# Example 10.1.3 / 1: Shared Capacity on a Small Network

Companion notebook for **10.1.3 Multicommodity Flows and Side Constraints** in the AMPL lab Documentation, section "Shared Capacity on a Small Network". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Shared Capacity on a Small Network

A plant ships bands and coils to two customers. The cheap route passes through a hub; direct links cost more. The plant-to-hub link can carry at most 2 coils (individual capacity) and at most 7 units of both products together (shared capacity).

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set CITIES;  # network nodes
set LINKS within CITIES cross CITIES;  # directed shipping links
set PRODS;  # commodities sharing the links

param supply {CITIES, PRODS} >= 0 default 0;   # units available
param demand {CITIES, PRODS} >= 0 default 0;   # units required
# each product's total supply equals its total demand
check {p in PRODS}:
    sum {i in CITIES} supply[i, p] = sum {j in CITIES} demand[j, p];

param cost {LINKS, PRODS} >= 0;       # cost per unit shipped
param capacity {LINKS, PRODS} >= 0;   # limit for one product
param cap_joint {LINKS} >= 0;         # limit for all products together

minimize TotalCost;  # terms are added by the arc declarations

node Balance {k in CITIES, p in PRODS}:  # per-product balance at each city
    net_in = demand[k, p] - supply[k, p];

# flow of product p on link (i, j), with its own limit
arc Ship {(i, j) in LINKS, p in PRODS} >= 0, <= capacity[i, j, p],
    from Balance[i, p], to Balance[j, p],
    obj TotalCost cost[i, j, p];

subject to JointCapacity {(i, j) in LINKS}:  # side constraint: all products share the link
    sum {p in PRODS} Ship[i, j, p] <= cap_joint[i, j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
links = [("plant", "hub"), ("hub", "east"), ("hub", "west"), ("plant", "east"), ("plant", "west")]
ampl.set["CITIES"] = ["plant", "hub", "east", "west"]
ampl.set["LINKS"] = links
ampl.set["PRODS"] = ["bands", "coils"]
ampl.param["supply"] = {("plant", "bands"): 6, ("plant", "coils"): 4}
ampl.param["demand"] = {("east", "bands"): 3, ("west", "bands"): 3, ("east", "coils"): 2, ("west", "coils"): 2}
ampl.param["cost"] = {
    ("plant", "hub", "bands"): 1, ("plant", "hub", "coils"): 1,
    ("hub", "east", "bands"): 1, ("hub", "east", "coils"): 1,
    ("hub", "west", "bands"): 1, ("hub", "west", "coils"): 1,
    ("plant", "east", "bands"): 3, ("plant", "east", "coils"): 5,
    ("plant", "west", "bands"): 4, ("plant", "west", "coils"): 6,
}
capacity = {(i, j, p): 10 for (i, j) in links for p in ["bands", "coils"]}
capacity["plant", "hub", "coils"] = 2
ampl.param["capacity"] = capacity
ampl.param["cap_joint"] = {(i, j): 7 if (i, j) == ("plant", "hub") else 10 for (i, j) in links}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
ship = ampl.get_data("Ship").to_dict()
print({key: value for key, value in ship.items() if value > 0})
print(ampl.get_data("{(i, j) in LINKS} sum {p in PRODS} Ship[i, j, p]").to_dict())
print(ampl.obj["TotalCost"].value())
ampl.param["cap_joint"]["plant", "hub"] = 10
ampl.solve(solver="highs")
print(ampl.get_data("{(i, j) in LINKS} sum {p in PRODS} Ship[i, j, p]").to_dict())
print(ampl.obj["TotalCost"].value())

HiGHS 1.14.0: optimal solution; objective 27
0 simplex iterations
0 barrier iterations
solved
{('hub', 'east', 'bands'): 2, ('hub', 'west', 'bands'): 3, ('hub', 'west', 'coils'): 2, ('plant', 'east', 'bands'): 1, ('plant', 'east', 'coils'): 2, ('plant', 'hub', 'bands'): 5, ('plant', 'hub', 'coils'): 2}
{('hub', 'east'): 2, ('hub', 'west'): 5, ('plant', 'east'): 3, ('plant', 'hub'): 7, ('plant', 'west'): 0}
27.0
HiGHS 1.14.0: optimal solution; objective 26
0 simplex iterations
0 barrier iterations
{('hub', 'east'): 3, ('hub', 'west'): 5, ('plant', 'east'): 2, ('plant', 'hub'): 8, ('plant', 'west'): 0}
26.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. `display` with an indexed `sum` shows the link totals, and `let` changes the shared capacity before the second `solve`:

In [6]:
%%writefile data.dat
set CITIES := plant hub east west;
set LINKS := (plant,hub) (hub,east) (hub,west) (plant,east) (plant,west);
set PRODS := bands coils;
param supply := [plant,bands] 6 [plant,coils] 4;
param demand := [east,bands] 3 [west,bands] 3 [east,coils] 2 [west,coils] 2;
param cost: bands coils :=
    plant hub    1 1
    hub   east   1 1
    hub   west   1 1
    plant east   3 5
    plant west   4 6;
param capacity default 10 := [plant,hub,coils] 2;
param cap_joint default 10 := [plant,hub] 7;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display {(i, j) in LINKS, p in PRODS: Ship[i, j, p] > 0} Ship[i, j, p];
display {(i, j) in LINKS} sum {p in PRODS} Ship[i, j, p];
display TotalCost;
let cap_joint['plant', 'hub'] := 10;
solve;
display {(i, j) in LINKS} sum {p in PRODS} Ship[i, j, p];
display TotalCost;

HiGHS 1.14.0: optimal solution; objective 27
0 simplex iterations
0 barrier iterations
Ship[i,j,p] :=
hub   east bands   2
hub   west bands   3
hub   west coils   2
plant east bands   1
plant east coils   2
plant hub  bands   5
plant hub  coils   2
;

sum{p in PRODS} Ship[i,j,p] :=
hub   east   2
hub   west   5
plant east   3
plant hub    7
plant west   0
;

TotalCost = 27

HiGHS 1.14.0: optimal solution; objective 26
0 simplex iterations
0 barrier iterations
sum{p in PRODS} Ship[i,j,p] :=
hub   east   3
hub   west   5
plant east   2
plant hub    8
plant west   0
;

TotalCost = 26

